# Module 14 — The AES S-box: Inverse plus Affine Transformation

**Mathematics of Cryptography · Volume 1**

---

## Overview

The AES SubBytes operation replaces each byte in the state with its image under the **S-box**, a fixed 256-entry substitution table. Unlike a random substitution, the AES S-box is constructed from two precisely chosen algebraic operations:

1. **Stage 1 — GF(2⁸) multiplicative inversion.**  
   Every nonzero byte `a` is replaced by its multiplicative inverse `a⁻¹` in GF(2⁸), the field of polynomials over GF(2) reduced modulo the AES irreducible polynomial `m(x) = x⁸ + x⁴ + x³ + x + 1`.  
   By convention `0x00⁻¹ = 0x00`.

2. **Stage 2 — Affine transformation over GF(2).**  
   The 8 bits of the inverse byte are mixed through a fixed linear map (an 8×8 binary circulant matrix) and then XORed with the constant `0x63`. Concretely, output bit `s_i` satisfies:

   $$s_i = b_i \oplus b_{(i+4)\%8} \oplus b_{(i+5)\%8} \oplus b_{(i+6)\%8} \oplus b_{(i+7)\%8} \oplus c_i$$

   where `b₀…b₇` are the bits of `a⁻¹` (LSB = index 0) and `(c₀…c₇) = (1,1,0,0,0,1,1,0)` are the bits of `0x63`.

### Why this construction?

- **Non-linearity** (from the inversion) defeats linear and differential cryptanalysis.
- **Algebraic simplicity** (invertibility of both stages) enables efficient AES decryption.
- **No fixed points** and no "zero sum" pairs — the affine constant `0x63` prevents `S-box(0x00) = 0x00`.

This notebook builds the S-box from scratch, verifies it against FIPS 197, and provides an interactive explorer.

In [ ]:
# ============================================================
# Section 2 — Setup: GF(2^8) arithmetic helpers
# ============================================================
# All arithmetic is modulo m(x) = x^8 + x^4 + x^3 + x + 1
# represented as the integer 0x11B = 283.

def xtime(a: int) -> int:
    """Multiply a by x (i.e. shift left by 1) in GF(2^8).
    If the high bit was set, XOR with 0x1B (the lower 8 bits of 0x11B)
    to reduce modulo m(x)."""
    result = (a << 1) & 0xFF
    if a & 0x80:          # high bit was set — reduction needed
        result ^= 0x1B
    return result


def gf_mul(a: int, b: int) -> int:
    """Multiply a and b in GF(2^8) using the 'peasant' method.
    For each bit of b (LSB first), if the bit is 1, XOR the current
    power-of-x multiple of a into the running result."""
    result = 0
    aa = a
    for i in range(8):
        if (b >> i) & 1:
            result ^= aa
        aa = xtime(aa)
    return result


def gf_inv(a: int) -> int:
    """Return the multiplicative inverse of a in GF(2^8).
    By convention gf_inv(0) = 0 (no true inverse for zero).
    We use a brute-force search over all 255 nonzero candidates."""
    if a == 0:
        return 0
    for b in range(1, 256):
        if gf_mul(a, b) == 1:
            return b
    raise ValueError(f"No inverse found for {a:#04x} — this should never happen")


# Affine constant: bits of 0x63, index 0 = LSB
CONST = [1, 1, 0, 0, 0, 1, 1, 0]

# Quick sanity checks
assert xtime(0x53) == 0xA6, "xtime(0x53) should be 0xA6"
assert gf_mul(0x53, 0xCA) == 0x01, "0x53 * 0xCA should equal 0x01 in GF(2^8)"
assert gf_inv(0x53) == 0xCA, "Inverse of 0x53 should be 0xCA"
assert gf_inv(0x00) == 0x00, "Inverse of 0 is 0 by convention"

print("GF(2^8) helpers verified.")
print(f"  xtime(0x53)       = {xtime(0x53):#04x}")
print(f"  gf_mul(0x53,0xCA) = {gf_mul(0x53, 0xCA):#04x}")
print(f"  gf_inv(0x53)      = {gf_inv(0x53):#04x}")
print(f"  CONST (0x63 bits) = {CONST}")

In [ ]:
# ============================================================
# Section 3 — Affine transformation
# ============================================================
# The affine map takes an 8-bit integer (the inverse byte),
# extracts its bits, mixes them with the circulant formula,
# XORs in the constant, and reassembles the result.

def affine(inv_byte: int) -> int:
    """Apply the AES affine transformation to inv_byte.
    Returns the transformed byte as an integer."""
    # Extract bits b0..b7 (LSB = index 0)
    b = [(inv_byte >> i) & 1 for i in range(8)]
    # Compute each output bit s_i
    s = [b[i] ^ b[(i+4)%8] ^ b[(i+5)%8] ^ b[(i+6)%8] ^ b[(i+7)%8] ^ CONST[i]
         for i in range(8)]
    # Reassemble into an integer
    result = 0
    for i in range(8):
        result |= s[i] << i
    return result


# ── Step-by-step trace for 0xCA ──────────────────────────────────────────
print("Step-by-step affine transformation of 0xCA (inverse of 0x53)")
print("=" * 60)

inv_byte = 0xCA
b = [(inv_byte >> i) & 1 for i in range(8)]
print(f"\nInput inv_byte  = {inv_byte:#04x} = {inv_byte:08b}b")
print(f"Bits (b0..b7)   = {b}  (index 0 = LSB)")
print(f"Constant CONST  = {CONST}  (bits of 0x63, index 0 = LSB)")
print()

s_bits = []
for i in range(8):
    si = b[i] ^ b[(i+4)%8] ^ b[(i+5)%8] ^ b[(i+6)%8] ^ b[(i+7)%8] ^ CONST[i]
    s_bits.append(si)
    print(f"s{i} = b{i}({b[i]}) ^ b{(i+4)%8}({b[(i+4)%8]}) ^"
          f" b{(i+5)%8}({b[(i+5)%8]}) ^ b{(i+6)%8}({b[(i+6)%8]}) ^"
          f" b{(i+7)%8}({b[(i+7)%8]}) ^ c{i}({CONST[i]}) = {si}")

result = affine(inv_byte)
print(f"\nOutput bits (s0..s7) = {s_bits}")
print(f"Assembled output     = {result:#04x} = {result:08b}b")
assert result == 0xED, f"Expected 0xED, got {result:#04x}"
print("\nCheck: affine(0xCA) == 0xED  ✓")

In [ ]:
# ============================================================
# Section 4 — The S-box function
# ============================================================

def sbox(a: int) -> int:
    """Compute the AES S-box value for byte a.
    Stage 1: GF(2^8) multiplicative inversion (0 -> 0).
    Stage 2: affine transformation over GF(2)."""
    return affine(gf_inv(a))


# ── Verification against known FIPS 197 values ───────────────────────────
known = [
    (0x00, 0x63),
    (0x53, 0xED),
    (0x7C, 0x10),
    (0xFF, 0x16),
]

print("FIPS 197 spot-checks")
print("-" * 40)
all_ok = True
for a, expected in known:
    got = sbox(a)
    status = "✓" if got == expected else f"FAIL (got {got:#04x})"
    print(f"  sbox({a:#04x}) = {got:#04x}   expected {expected:#04x}  {status}")
    if got != expected:
        all_ok = False

assert all_ok, "One or more FIPS 197 spot-checks failed!"
print("\nAll spot-checks passed.")

In [ ]:
# ============================================================
# Section 5 — Full 256-entry AES S-box table
# ============================================================

SBOX = [sbox(i) for i in range(256)]

# Print in the traditional 16×16 grid format (matches FIPS 197 Table 4)
print("AES S-box (rows = high nibble, columns = low nibble of input)")
print("     ", end="")
for col in range(16):
    print(f" _{col:X}", end="")
print()
print("    " + "-" * 52)
for row in range(16):
    print(f" {row:X}_ |", end="")
    for col in range(16):
        idx = (row << 4) | col
        print(f" {SBOX[idx]:02X}", end="")
    print()

print()
# Additional FIPS 197 spot-checks from the standard (a selection)
fips_spot = {
    0x00: 0x63, 0x01: 0x7C, 0x02: 0x77, 0x03: 0x7B,
    0x10: 0xCA, 0x53: 0xED, 0x63: 0xFB, 0x7C: 0x10,
    0xAB: 0x62, 0xF0: 0x8C, 0xFE: 0xBB, 0xFF: 0x16,
}
print("Additional FIPS 197 spot-checks:")
all_ok = True
for a, expected in sorted(fips_spot.items()):
    got = SBOX[a]
    status = "✓" if got == expected else f"FAIL (got {got:#04x})"
    print(f"  SBOX[{a:#04x}] = {got:#04x}  {status}")
    if got != expected:
        all_ok = False
assert all_ok, "FIPS 197 table mismatch!"
print("\nAll FIPS 197 spot-checks passed.")

In [ ]:
# ============================================================
# Section 6 — Verify: no fixed points and bijection
# ============================================================

# A fixed point is a byte a where sbox(a) == a.
# The affine constant 0x63 is specifically chosen to eliminate fixed points.

fixed_points = [a for a in range(256) if SBOX[a] == a]
print(f"Fixed points of the S-box: {fixed_points}")
assert len(fixed_points) == 0, f"Unexpected fixed points: {fixed_points}"
print("No fixed points — confirmed.\n")

# A bijection means the S-box is a permutation:
# every output value appears exactly once.
output_values = sorted(set(SBOX))
assert len(output_values) == 256, "S-box is NOT a bijection — some output missing!"
print(f"Number of distinct output values: {len(output_values)}")
print("S-box is a bijection (permutation of all 256 byte values) — confirmed.\n")

# Verify also that sbox(sbox(a)) != a for all a (no period-2 cycles would be a
# concern; in practice the inverse S-box is a distinct mapping)
double_fixed = [a for a in range(256) if SBOX[SBOX[a]] == a]
print(f"Bytes where sbox(sbox(a)) == a (period-2 cycles): {len(double_fixed)} found")
# There are some period-2 cycles — that is normal and expected for this S-box.
# The important property is that the S-box itself has no fixed points.
print("(Period-2 cycles are normal; the key property is no fixed points.)")

In [ ]:
# ============================================================
# Section 7 — Worked example: full trace of S-box(0x53)
# ============================================================

print("Full step-by-step trace: S-box(0x53)")
print("=" * 60)

a = 0x53
print(f"\nInput byte a = {a:#04x} = {a:08b}b")

# Stage 1: GF(2^8) inversion
print("\n--- Stage 1: GF(2^8) multiplicative inversion ---")
inv = gf_inv(a)
print(f"  We need b such that gf_mul(0x53, b) == 0x01.")
print(f"  By the Extended Euclidean Algorithm (or lookup):")
print(f"  0x53^(-1) = {inv:#04x} = {inv:08b}b")
verify_product = gf_mul(a, inv)
print(f"  Verify: gf_mul({a:#04x}, {inv:#04x}) = {verify_product:#04x}  ✓" if verify_product == 1 else "FAIL")

# Stage 2: Affine transformation
print("\n--- Stage 2: Affine transformation ---")
b_bits = [(inv >> i) & 1 for i in range(8)]
print(f"  Bits of {inv:#04x} (b0..b7, LSB first): {b_bits}")
print(f"  Constant CONST (c0..c7, LSB first):      {CONST}")
print()

s_bits = []
for i in range(8):
    si = b_bits[i] ^ b_bits[(i+4)%8] ^ b_bits[(i+5)%8] ^ b_bits[(i+6)%8] ^ b_bits[(i+7)%8] ^ CONST[i]
    s_bits.append(si)
    print(f"  s{i} = b{i}({b_bits[i]}) ^ b{(i+4)%8}({b_bits[(i+4)%8]}) "
          f"^ b{(i+5)%8}({b_bits[(i+5)%8]}) ^ b{(i+6)%8}({b_bits[(i+6)%8]}) "
          f"^ b{(i+7)%8}({b_bits[(i+7)%8]}) ^ c{i}({CONST[i]}) = {si}")

output = 0
for i in range(8):
    output |= s_bits[i] << i

print(f"\n  Output bits (s0..s7): {s_bits}")
print(f"  Assembled output:     {output:#04x} = {output:08b}b")
assert output == 0xED, f"Expected 0xED, got {output:#04x}"
print(f"\nResult: S-box(0x53) = {output:#04x} = 0xED  ✓  (matches FIPS 197)")

In [ ]:
# ============================================================
# Section 8 — Interactive explorer: sbox_explore(hex_val)
# ============================================================

def sbox_explore(hex_val):
    """Pretty-print the full two-stage S-box computation for any input.

    Parameters
    ----------
    hex_val : int or str
        The input byte, e.g. 0x7C or '0x7C' or 124.

    Examples
    --------
    >>> sbox_explore(0x53)
    >>> sbox_explore('0xAB')
    >>> sbox_explore(0)
    """
    if isinstance(hex_val, str):
        a = int(hex_val, 16) if hex_val.startswith(('0x', '0X')) else int(hex_val, 16)
    else:
        a = int(hex_val)

    if not 0 <= a <= 255:
        print(f"Error: input must be in range 0x00–0xFF, got {a}")
        return

    print(f"S-box exploration for input {a:#04x} ({a:08b}b, decimal {a})")
    print("-" * 55)

    # Stage 1
    inv = gf_inv(a)
    if a == 0:
        print(f"Stage 1 (GF inversion): {a:#04x} -> {inv:#04x}  [special case: 0 maps to 0 by convention]")
    else:
        verify = gf_mul(a, inv)
        print(f"Stage 1 (GF inversion): {a:#04x} -> {inv:#04x}")
        print(f"  Verify: gf_mul({a:#04x}, {inv:#04x}) = {verify:#04x}  {'✓' if verify == 1 else 'FAIL'}")

    # Stage 2
    b_bits = [(inv >> i) & 1 for i in range(8)]
    s_bits = [
        b_bits[i] ^ b_bits[(i+4)%8] ^ b_bits[(i+5)%8] ^ b_bits[(i+6)%8] ^ b_bits[(i+7)%8] ^ CONST[i]
        for i in range(8)
    ]
    output = 0
    for i in range(8):
        output |= s_bits[i] << i

    print(f"Stage 2 (affine):       {inv:#04x} -> {output:#04x}")
    print(f"  Bits of inverse (b0..b7): {b_bits}")
    print(f"  Output bits  (s0..s7):    {s_bits}")
    print(f"\nResult: S-box({a:#04x}) = {output:#04x} ({output:08b}b, decimal {output})")


# ── Try a few examples ───────────────────────────────────────────────────
for test_val in [0x00, 0x53, 0x7C, 0xFF, 0xAB]:
    sbox_explore(test_val)
    print()

In [ ]:
# ============================================================
# Section 9 — The 0x00 special case
# ============================================================
# Understanding why S-box(0x00) = 0x63 requires examining both stages.

print("Why S-box(0x00) = 0x63")
print("=" * 55)

print("""
Mathematical situation:
  In GF(2^8), multiplication by 0x00 always yields 0x00.
  There is no byte b satisfying 0x00 * b = 0x01,
  so 0x00 has NO multiplicative inverse in the true field sense.

AES design decision:
  The FIPS 197 specification defines the inverse map as:
    inv(0) = 0          (a special-case mapping, not a true inverse)
    inv(a) = a^(-1)     for a != 0
  This makes the inverse map a bijection on {0, ..., 255},
  which is necessary for the S-box to be invertible.
""")

# Trace through both stages for 0x00
a = 0x00
inv = gf_inv(a)   # returns 0 by convention
print(f"Stage 1: gf_inv({a:#04x}) = {inv:#04x}  [special-case convention]")

b_bits = [(inv >> i) & 1 for i in range(8)]
print(f"\nBits of inv ({inv:#04x}): {b_bits}  (all zeros)")
print(f"Constant CONST (0x63): {CONST}")
print()
print("Since all b_i = 0, each output bit s_i = 0 XOR 0 XOR 0 XOR 0 XOR 0 XOR c_i = c_i")
print(f"So the output is simply the constant 0x63 = {0x63:08b}b")

result = sbox(0x00)
print(f"\nS-box(0x00) = {result:#04x}")
assert result == 0x63
print("Confirmed: S-box(0x00) = 0x63  ✓")

print("""
Significance:
  If the special case were S-box(0x00) = 0x00, then an all-zero
  AES state would encrypt to an all-zero state regardless of the key —
  a catastrophic weakness. The constant 0x63 prevents this.
  More precisely, the affine constant was chosen so that the S-box
  has no fixed points (S-box(a) != a for all a) and no 'opposite'
  fixed points (S-box(a) != ~a for all a).
""")

## Section 10 — Summary and Bridge to Module 15

### What we built

Starting from the GF(2⁸) arithmetic helpers in Module 12 and 13, we constructed the complete AES S-box:

| Step | Operation | Key property |
|------|-----------|-------------|
| 1 | GF(2⁸) multiplicative inversion | Provides non-linearity; bijection on {0…255} |
| 2 | Affine transformation (matrix + XOR 0x63) | Preserves invertibility; eliminates fixed points |
| Combined | `sbox(a) = affine(gf_inv(a))` | Bijection; non-linear; algebraically structured |

### Key results verified

- `S-box(0x00) = 0x63` — the constant 0x63 drives this when all inverse bits are 0.
- `S-box(0x53) = 0xED` — the canonical worked example from FIPS 197.
- `S-box(0x7C) = 0x10` and `S-box(0xFF) = 0x16` — additional FIPS 197 values.
- **No fixed points**: `S-box(a) ≠ a` for all 256 values of `a`.
- **Bijection**: all 256 output values are distinct.

### Why the S-box is not just a random table

A random permutation would provide confusion but give no algebraic guarantees about resistance to differential or linear cryptanalysis. The AES S-box was deliberately constructed so that:

- The maximum differential probability is as low as possible (2⁻⁶ per S-box).
- The maximum linear bias is as low as possible.
- Both stages are individually invertible (enabling efficient decryption).
- The construction is fully transparent and published (no hidden backdoor).

### Bridge to Module 15 — The Inverse S-box (InvSubBytes)

AES decryption requires undoing the S-box. Because both stages are invertible, the **inverse S-box** is constructed by:

1. **Invert the affine transformation**: apply the inverse affine map (a different matrix and constant).
2. **Invert Stage 1**: compute the GF(2⁸) multiplicative inverse again (since `(a⁻¹)⁻¹ = a` for nonzero elements).

Module 15 derives the InvSubBytes table, verifies that `InvSBox(SBox(a)) = a` for all bytes, and shows how it fits into the AES decryption round function alongside InvShiftRows, InvMixColumns, and AddRoundKey.

---

*Mathematics of Cryptography · Module 14 · The AES S-box: Inverse plus Affine Transformation*